# Visualization

Visualization notebook for platform, topic, and time-based analysis across the processed datasets.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "dataset").exists() and (candidate / "src").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PREPROCESS_DIR = PROJECT_ROOT / "dataset" / "processed_data"
UNION_FILE = PREPROCESS_DIR / "social_media_analysis_ready.csv"
SOURCE_METRICS_FILE = PREPROCESS_DIR / "dashboard_source_metrics.csv"
MONTHLY_METRICS_FILE = PREPROCESS_DIR / "dashboard_monthly_metrics.csv"
TOPIC_METRICS_FILE = PREPROCESS_DIR / "dashboard_topic_metrics.csv"
CATALOG_PATH = PREPROCESS_DIR / "dataset_catalog.json"

with CATALOG_PATH.open("r", encoding="utf-8") as handle:
    catalog = json.load(handle)

source_metrics = pd.read_csv(SOURCE_METRICS_FILE)
monthly_metrics = pd.read_csv(MONTHLY_METRICS_FILE)
topic_metrics = pd.read_csv(TOPIC_METRICS_FILE)
union_df = pd.read_csv(UNION_FILE, parse_dates=["published_at", "trend_date"])

len(union_df), source_metrics.head()


In [ ]:
pd.DataFrame(catalog["datasets"])[["title", "processed_rows", "platforms", "date_columns", "processed_file"]]


In [ ]:
platform_mix = (
    source_metrics.groupby(["platform"], as_index=False)[["record_count", "total_engagement"]]
    .sum()
    .sort_values("record_count", ascending=False)
)

plt.figure(figsize=(10, 5))
sns.barplot(data=platform_mix, x="record_count", y="platform", color="#254D6E")
plt.title("Record coverage by platform")
plt.xlabel("Records")
plt.ylabel("Platform")
plt.tight_layout()
plt.show()


In [ ]:
dated_monthly = monthly_metrics.loc[monthly_metrics["year_month"] != "Unknown"].copy()
trend = dated_monthly.groupby("year_month", as_index=False)[["record_count", "total_engagement"]].sum()

plt.figure(figsize=(12, 5))
sns.lineplot(data=trend, x="year_month", y="record_count", marker="o", color="#D96F32", linewidth=2.2)
plt.xticks(rotation=45, ha="right")
plt.title("Cross-source activity trend")
plt.xlabel("Month")
plt.ylabel("Records")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
sentiment_mix = (
    source_metrics.loc[source_metrics["sentiment"] != "Unknown"]
    .groupby("sentiment", as_index=False)["record_count"]
    .sum()
    .sort_values("record_count", ascending=False)
)

if not sentiment_mix.empty:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=sentiment_mix, x="sentiment", y="record_count", palette="crest")
    plt.title("Sentiment coverage across text datasets")
    plt.xlabel("Sentiment")
    plt.ylabel("Records")
    plt.tight_layout()
    plt.show()


In [ ]:
top_topics = topic_metrics.sort_values(["record_count", "total_engagement"], ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_topics, x="record_count", y="primary_topic", hue="platform")
plt.title("Most visible explicit topics across tagged datasets")
plt.xlabel("Records")
plt.ylabel("Topic")
plt.tight_layout()
plt.show()
